<a href="https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis.** One row = one content item (`content_hash_id`) for one client (`client_hash_id`), representing that item's search performance over a defined 60-day look-back window anchored to the prediction date.

**Time windows.** We split each item's history into two non-overlapping 30-day bands:

| Band | Days relative to prediction date | Purpose |
|---|---|---|
| Feature window | [t−60, t−30) | Known before prediction — safe features |
| Label window | [t−30, t) | What we're predicting — never a feature |

**Label definition.** `is_declining = 1` if impressions in the label window are less than 80 % of impressions in the feature window (> 20 % month-on-month decline), else 0. We require at least 100 impressions in the feature window (`imp_prev30 ≥ 100`) for the ratio to be reliable.

In [8]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# HF token — read from Colab Secrets only (never paste a token in a cell of a public repo)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

# Warehouse connection
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

TARGET_MONTH = "2026-03"   # mid-panel month — never change to the final month (2026-06)

print("✓ Warehouse connection established")

# 1a. Grain check — confirm one row per (report_date, client_hash_id, content_hash_id) in TARGET_MONTH
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id)                                         AS unique_content,
        COUNT(DISTINCT client_hash_id)                                          AS unique_clients,
        COUNT(DISTINCT report_date || '_' || client_hash_id || '_' || content_hash_id)
                                                                                AS unique_triplets
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{TARGET_MONTH}'
""").df()

total  = grain_check.iloc[0]['total_rows']
unique = grain_check.iloc[0]['unique_triplets']
print(f"Total rows in {TARGET_MONTH} : {total:,}")
print(f"Unique (date × client × content): {unique:,}")
print(f"Grain holds (total == unique): {total == unique}")

print()

# 1b. Date-span check
span = con.sql(f"""
    SELECT
        COUNT(*)              AS row_count,
        MIN(report_date)      AS min_date,
        MAX(report_date)      AS max_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{TARGET_MONTH}'
""").df()

print(f"Date range : {span.iloc[0]['min_date']} → {span.iloc[0]['max_date']}")
print(f"Distinct dates: {span.iloc[0]['distinct_dates']}  (expected 31 for March 2026)")

print()

# 1c. Availability — how many (content × client) windows have imp_prev30 >= 100
avail = con.sql(f"""
    SELECT
        COUNT(*)                                        AS total_windows,
        COUNT(*) FILTER (WHERE imp_prev30 >= 100)       AS available,
        ROUND(
            COUNT(*) FILTER (WHERE imp_prev30 >= 100)
            * 100.0 / COUNT(*), 1
        )                                               AS availability_pct
    FROM (
        SELECT
            f.content_hash_id,
            f.client_hash_id,
            SUM(
                CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END
            ) AS imp_prev30
        FROM {TABLES['fact_daily']} f,
             (SELECT MAX(report_date) AS end_d
              FROM {TABLES['fact_daily']}
              WHERE strftime(report_date, '%Y-%m') = '{TARGET_MONTH}') b
        WHERE f.report_date >  b.end_d - INTERVAL 60 DAY
          AND f.report_date <= b.end_d
          AND strftime(f.report_date, '%Y-%m') <= '{TARGET_MONTH}'
        GROUP BY 1, 2
    )
""").df()

print(f"Total (content × client) windows : {avail.iloc[0]['total_windows']:,}")
print(f"With imp_prev30 ≥ 100            : {avail.iloc[0]['available']:,}  ({avail.iloc[0]['availability_pct']} %)")


✓ Warehouse connection established


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows in 2026-03 : 9,841,378
Unique (date × client × content): 9,841,378
Grain holds (total == unique): True



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Date range : 2026-03-01 00:00:00 → 2026-03-31 00:00:00
Distinct dates: 31  (expected 31 for March 2026)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total (content × client) windows : 349,411.0
With imp_prev30 ≥ 100            : 82,025.0  (23.5 %)


## 2. Fields: feature / label / context / excluded

Every field we plan to touch goes in exactly one bucket.

### Features — knowable before the prediction moment

| Field | Source | Window | Why it's safe |
|---|---|---|---|
| `imp_prev30` | `fact_daily` | [t−60, t−30) | Historical impressions, no overlap with label |
| `clk_prev30` | `fact_daily` | [t−60, t−30) | Historical clicks, same window |
| `pos_prev30` | `fact_daily` | [t−60, t−30) | Average GSC position, same window |
| `visible_queries` | `fact_query_90d` | Fixed 90-day snapshot | Number of distinct queries driving impressions — historical |
| `top_query_share` | `fact_query_90d` | Fixed 90-day snapshot | Top query impressions / total impressions — query concentration |

**Knowability test:** every feature above satisfies "could I have known this value on the day before I make the prediction, without knowing the outcome?"

### Label / proxy — what we predict (never a feature)

| Field | Derived from | Why excluded from features |
|---|---|---|
| `is_declining` | `imp_last30 < 0.8 × imp_prev30` | This is the outcome we predict |
| `imp_last30` | `fact_daily` label window [t−30, t) | Directly computes the label |

### Context — for grouping, joining, splitting only

| Field | Use |
|---|---|
| `content_hash_id` | Joins to `dim_content`; groups rows per content item |
| `client_hash_id` | Joins to `dim_clients`; used for grouped train/test splits |
| `report_date` | Defines time windows — never passed to the model |

### Excluded — with reasons

| Field | Why excluded |
|---|---|
| `trend_pct` / `trend_direction` (starter CSV) | Computed from the same impression comparison as the label — pure leakage |
| Any product flag (`health_score`, `needs_*_fix`) | FlyRank rule-based outputs; encoding a decision already made — circular result |
| `imp_last30` as a feature | Future information (the label window) |
| GA4 columns before `ga4_data_available = TRUE` | Zero-filled placeholders, not real zeros |

*Verification: the code cell below confirms the leakage signal by adding a deliberately leaky column, watching performance jump, then removing it.*

In [9]:
from scipy.stats import pointbiserialr

FEATURE_COLS = [
    'feature_imp_prev30',
    'feature_clk_prev30',
    'feature_pos_prev30',
    'feature_visible_queries',
    'feature_top_query_share',
]

# Shared subqueries
_windows = f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_clicks    ELSE 0 END) AS clk_prev30,
        AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_avg_position ELSE 0 END) AS pos_prev30,
        SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_last30
    FROM {TABLES['fact_daily']} f,
         (SELECT MAX(report_date) AS end_d
          FROM {TABLES['fact_daily']}
          WHERE strftime(report_date, '%Y-%m') = '{TARGET_MONTH}') b
    WHERE f.report_date >  b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
      AND strftime(f.report_date, '%Y-%m') <= '{TARGET_MONTH}'
    GROUP BY 1, 2
    HAVING imp_prev30 >= 100
"""

_queries = f"""
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        MAX(impressions_90d)                   AS top_query_impressions,
        SUM(impressions_90d)                   AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
"""

# ── 2a. Honest model (no leakage) ────────────────────────────────────────────
honest = con.sql(f"""
    SELECT
        w.imp_prev30                                                              AS feature_imp_prev30,
        w.clk_prev30                                                              AS feature_clk_prev30,
        w.pos_prev30                                                              AS feature_pos_prev30,
        q.visible_queries                                                         AS feature_visible_queries,
        CASE WHEN q.kept_impressions > 0
             THEN q.top_query_impressions / q.kept_impressions ELSE 0 END         AS feature_top_query_share,
        CASE WHEN w.imp_last30 < 0.8 * w.imp_prev30 THEN 1 ELSE 0 END            AS is_declining
    FROM ({_windows}) w
    JOIN ({_queries})  q ON w.content_hash_id = q.content_hash_id
    WHERE q.kept_impressions > 0
    LIMIT 5000
""").df()

def mean_abs_corr(df, cols, label='is_declining'):
    scores = []
    for c in cols:
        if c in df and df[c].std() > 0:
            r, _ = pointbiserialr(df[label], df[c])
            scores.append(abs(r))
    return float(np.mean(scores)) if scores else 0.0

baseline = mean_abs_corr(honest, FEATURE_COLS)
print(f"Honest model  — avg |point-biserial r|: {baseline:.4f}")

# ── 2b. Leaky model (add trend_pct, which is algebraically equivalent to the label) ──
leaky = con.sql(f"""
    SELECT
        w.imp_prev30                                                              AS feature_imp_prev30,
        w.clk_prev30                                                              AS feature_clk_prev30,
        w.pos_prev30                                                              AS feature_pos_prev30,
        q.visible_queries                                                         AS feature_visible_queries,
        CASE WHEN q.kept_impressions > 0
             THEN q.top_query_impressions / q.kept_impressions ELSE 0 END         AS feature_top_query_share,
        -- DELIBERATE LEAKAGE: trend_pct is (imp_last30 - imp_prev30) / imp_prev30
        -- It encodes the same information as is_declining without a threshold.
        CASE WHEN w.imp_prev30 > 0
             THEN (w.imp_last30 - w.imp_prev30) * 100.0 / w.imp_prev30
             ELSE 0 END                                                            AS leaky_trend_pct,
        CASE WHEN w.imp_last30 < 0.8 * w.imp_prev30 THEN 1 ELSE 0 END            AS is_declining
    FROM ({_windows}) w
    JOIN ({_queries})  q ON w.content_hash_id = q.content_hash_id
    WHERE q.kept_impressions > 0
    LIMIT 5000
""").df()

leaky_score = mean_abs_corr(leaky, FEATURE_COLS + ['leaky_trend_pct'])
print(f"Leaky model   — avg |point-biserial r|: {leaky_score:.4f}")
print(f"Δ from leakage: +{leaky_score - baseline:.4f}  ({(leaky_score - baseline) / baseline * 100:.1f} % jump)")
print()
print("⚠  leaky_trend_pct is algebraically equivalent to the label — it caused the jump.")
print("✓  Removing it. Only honest features go forward.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest model  — avg |point-biserial r|: 0.0720


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaky model   — avg |point-biserial r|: 0.0963
Δ from leakage: +0.0242  (33.7 % jump)

⚠  leaky_trend_pct is algebraically equivalent to the label — it caused the jump.
✓  Removing it. Only honest features go forward.


## 3. Verify it with queries (grain, counts, missing values, windows)

**What we verify:**
1. The five honest features can be computed with no temporal leakage (feature frame materialises cleanly).
2. Each feature has a stable, non-degenerate distribution (no all-zero columns, no divide-by-zero blow-ups).
3. The join to `fact_query_90d` does not silently shrink the dataset to near-zero.

*Missingness notes:*
- `visible_queries` and `top_query_share` require a join to the query table. Content items with no query rows are dropped — the code cell measures the drop rate.
- `pos_prev30` is averaged over only the rows that exist; if a content item has no impressions in the feature window it is already excluded by `HAVING imp_prev30 ≥ 100`.

In [10]:
# 3a. Build the feature frame for TARGET_MONTH (sample of 1 000 rows for speed)
feature_frame = con.sql(f"""
    SELECT
        w.content_hash_id,
        w.client_hash_id,
        -- Feature 1 — historical impressions [t-60, t-30)
        w.imp_prev30                                                          AS feature_imp_prev30,
        -- Feature 2 — historical clicks [t-60, t-30)
        w.clk_prev30                                                          AS feature_clk_prev30,
        -- Feature 3 — average GSC position [t-60, t-30)
        w.pos_prev30                                                          AS feature_pos_prev30,
        -- Feature 4 — distinct queries driving impressions (90-day snapshot)
        q.visible_queries                                                     AS feature_visible_queries,
        -- Feature 5 — query concentration: top-query impressions / total impressions
        CASE WHEN q.kept_impressions > 0
             THEN q.top_query_impressions / q.kept_impressions ELSE 0 END     AS feature_top_query_share
    FROM ({_windows}) w
    JOIN ({_queries})  q ON w.content_hash_id = q.content_hash_id
    WHERE q.kept_impressions > 0
    LIMIT 1000
""").df()

# 3b. How much did the join to the query table shrink the pool?
windows_only = con.sql(f"""
    SELECT COUNT(*) AS n FROM ({_windows})
""").df().iloc[0,0]

print(f"(content × client) windows with imp_prev30 ≥ 100 : {windows_only:,}")
print(f"After joining to fact_query_90d (sample cap 1 000): {len(feature_frame):,}")
print()

# 3c. Feature statistics
print("Feature statistics:")
print(feature_frame[FEATURE_COLS].describe().round(2).to_string())
print()

# 3d. Null counts
null_counts = feature_frame[FEATURE_COLS].isnull().sum()
print("Null counts per feature:")
print(null_counts.to_string())
print()

# 3e. Sample rows
print("Sample rows (first 5):")
print(feature_frame[['content_hash_id', 'client_hash_id'] + FEATURE_COLS].head().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(content × client) windows with imp_prev30 ≥ 100 : 82,025
After joining to fact_query_90d (sample cap 1 000): 1,000

Feature statistics:
       feature_imp_prev30  feature_clk_prev30  feature_pos_prev30  feature_visible_queries  feature_top_query_share
count             1000.00             1000.00             1000.00                  1000.00                  1000.00
mean              2463.96                8.48                6.92                    25.81                     0.40
std               4238.28               21.00                5.80                    58.96                     0.27
min                101.00                0.00                0.18                     1.00                     0.03
25%                320.75                0.00                3.12                     4.00                     0.19
50%               1039.50                3.00                4.94                    10.00                     0.32
75%               2608.75               10.00      

## 4. Data limits

What can this data never tell us? Six things worth naming:

1. **Unbalanced panel — history depth varies per client.** `dim_clients.gsc_data_start` differs across 104 clients; some have only 1–2 months. A global calendar window silently drops recent-joiners or mixes shallow and deep histories. *Mitigation: require a minimum history threshold (e.g. 6+ months) and prefer per-client windows.*

2. **GSC-only early rows before GA4 onboarding.** Rows before a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. Those zeros are not real engagement signals. *Mitigation: filter on `ga4_data_available = TRUE` before using any GA4-derived feature.*

3. **Fixed 90-day query window overlaps the label window.** `fact_content_query_90d` is a fixed snapshot. If the label lives in the last 30 days of that window, any query metric that aggregates the full 90 days is partially future information. *Mitigation: use only `*_prev30`-style aggregations from the query table, or confirm there is no date overlap with the label window.*

4. **Aggregation floor — daily grain, no intra-day signals.** The fact table is one row per date. Rapid same-day rank changes are invisible. *Mitigation: acknowledge the model predicts 30-day trends, not daily movements.*

5. **Query privacy thresholds suppress long-tail queries.** Low-volume queries are anonymised or dropped. `visible_queries` and `rare_share` are lower bounds on true query diversity. *Mitigation: treat query metrics as relative, directional signals — not absolute counts.*

6. **No external factors.** Algorithm updates, seasonality, and competitor activity are unobserved. Observed declines may have causes outside the data. *Mitigation: use careful claim language — "associated with", "directional", "measured" — never "predicted" or "caused".*

*The code cell below verifies the most important structural limit: how many clients have less than 6 months of usable history.*

In [11]:
# 4a. History depth per client — how unbalanced is the panel?
client_history = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS days_in_warehouse,
        DATEDIFF('month', MIN(report_date), MAX(report_date)) AS approx_months
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
    ORDER BY approx_months
""").df()

bins = [0, 3, 6, 12, 999]
labels = ['< 3 mo', '3–6 mo', '6–12 mo', '12+ mo']
client_history['history_bucket'] = pd.cut(
    client_history['approx_months'], bins=bins, labels=labels, right=False
)

print("Client history distribution:")
print(client_history['history_bucket'].value_counts().sort_index().to_string())
print()
print(f"Clients with < 6 months of history: "
      f"{(client_history['approx_months'] < 6).sum()} / {len(client_history)}")
print()

# 4b. GA4 availability by client — which clients have ga4_data_available rows
ga4_avail = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*)                        AS rows,
        COUNT(DISTINCT client_hash_id)  AS clients
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{TARGET_MONTH}'
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available DESC
""").df()

print("GA4 availability in target month:")
print(ga4_avail.to_string(index=False))
print()
print("✓ Rows where ga4_data_available = FALSE should be filtered before using GA4-derived features.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Client history distribution:
history_bucket
< 3 mo     11
3–6 mo     20
6–12 mo    30
12+ mo      9

Clients with < 6 months of history: 31 / 70



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 availability in target month:
 ga4_data_available    rows  clients
               True  413966       41
              False 6408671       43
               <NA> 3018741       22

✓ Rows where ga4_data_available = FALSE should be filtered before using GA4-derived features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.